# Hyperparameter search with Katib, tracked in MLflow

Goal:

- Search YOLO training hyperparameters with Katib instead of guessing them.
- Each trial is a **pod** running the training image, not a notebook cell.
- Track every trial in MLflow so the search is comparable with the runs from
  the earlier notebooks.

Katib owns the search; MLflow owns the record. From image `v0.3.0` the trials
log themselves: `train.py` opens its own MLflow run and reports per-epoch
metrics, system metrics (cpu, memory, disk, network and gpu) and the run
artifacts while it trains. The notebook only has to pass the tracking server in
through the trial spec.

Katib still scrapes stdout for the objective, so the two are independent -- if
the tracking server is unreachable the trial logs a warning and trains anyway.


## Environment

Install dependencies.


In [ ]:
# pip install
%pip install -q -U kubeflow-katib kubernetes mlflow

Inspect environment.

- Train image: 099139718958.dkr.ecr.ca-central-1.amazonaws.com/kubeflow-yolo-train:v0.3.0


In [ ]:
# inspect env
import os
from pathlib import Path

from kubeflow.katib import KatibClient
from kubernetes.client import V1ObjectMeta
from kubeflow.katib import V1beta1AlgorithmSpec
from kubeflow.katib import V1beta1Experiment
from kubeflow.katib import V1beta1ExperimentSpec
from kubeflow.katib import V1beta1FeasibleSpace
from kubeflow.katib import V1beta1ObjectiveSpec
from kubeflow.katib import V1beta1ParameterSpec
from kubeflow.katib import V1beta1TrialTemplate

# the profile namespace this notebook runs in; trials land here too
NAMESPACE = os.environ.get("NAMESPACE", "kubeflow-yolo")

ACCOUNT = "099139718958"
REGION = "ca-central-1"
# gpu image
IMAGE = f"{ACCOUNT}.dkr.ecr.{REGION}.amazonaws.com/kubeflow-yolo-train:v0.3.0"

EXPERIMENT_NAME = "kubeflow-yolo-plate-mlflow"

print("namespace  ", NAMESPACE)
print("image      ", IMAGE)

### MLflow tracking

Point at the in-cluster tracking server. These two values are passed into every
trial pod as environment variables further down, which is how the trials find
the server.


In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

# in-cluster service; port 80 is the chart's service port
MLFLOW_TRACKING_URI = os.environ.get(
    "MLFLOW_TRACKING_URI", "http://mlflow.kubeflow.svc.cluster.local"
)
MLFLOW_EXPERIMENT = os.environ.get("MLFLOW_EXPERIMENT", EXPERIMENT_NAME)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print("tracking uri ", MLFLOW_TRACKING_URI)
print("experiment   ", MLFLOW_EXPERIMENT)

---

## Hyperparameters

Four parameters, chosen for a single-class small-object detector:

| Parameter | Range       | Why                                                          |
| --------- | ----------- | ------------------------------------------------------------ |
| `lr0`     | 1e-4 – 2e-2 | the highest-impact single knob                               |
| `batch`   | 4 – 24      | gradient noise; interacts with `lr0`                         |
| `box`     | 5 – 12      | box loss gain — with one class, localization is what matters |
| `scale`   | 0.3 – 0.9   | plate size varies with camera distance                       |


In [ ]:
# define parameters
parameters = [
    V1beta1ParameterSpec(
        name="lr0",
        parameter_type="double",
        feasible_space=V1beta1FeasibleSpace(min="0.0001", max="0.02"),
    ),
    V1beta1ParameterSpec(
        name="batch",
        parameter_type="int",
        feasible_space=V1beta1FeasibleSpace(min="4", max="24", step="4"),
    ),
    V1beta1ParameterSpec(
        name="box",
        parameter_type="double",
        feasible_space=V1beta1FeasibleSpace(min="5.0", max="12.0"),
    ),
    V1beta1ParameterSpec(
        name="scale",
        parameter_type="double",
        feasible_space=V1beta1FeasibleSpace(min="0.3", max="0.9"),
    ),
]

for p in parameters:
    fs = p.feasible_space
    print(f"{p.name:8} {p.parameter_type:7} {fs.min} .. {fs.max}")

### Objective

- `train.py` returns `name=value` lines at the end of a run;
- Katib's default collector scrapes stdout for them.

```
mAP50=0.812345
mAP50-95=0.643210
precision=0.887654
recall=0.791234
```

- `mAP50-95`: the stricter metric


In [ ]:
objective = V1beta1ObjectiveSpec(
    type="maximize",
    goal=0.85,  # stop early if a trial reaches this
    objective_metric_name="mAP50-95",
    additional_metric_names=["mAP50", "precision", "recall"],
)

# bayesian optimization: strategy used to find the best input values
algorithm = V1beta1AlgorithmSpec(algorithm_name="bayesianoptimization")

print("objective   maximize", objective.objective_metric_name, "goal", objective.goal)
print("algorithm  ", algorithm.algorithm_name)

---

## Trials

Each trial is a `batch/v1` Job.

Notes on the pod spec:

- `restartPolicy: Never`: a failed trial will not retried.
- `sidecar.istio.io/inject: "false"`: disable the mesh sidecar
- One GPU per trial on `g5.xlarge` (1x A10G).
  - `workload-class=gpu:NoSchedule`: tainted with gpu
  - `nvidia.com/gpu:NoSchedule`: tained with nvidia device plugin
  - `workload-class: gpu`: select gpu
- `parallelTrialCount`: control parallel


In [ ]:
trial_parameters = [
    {"name": "lr0", "description": "initial learning rate", "reference": "lr0"},
    {"name": "batch", "description": "batch size", "reference": "batch"},
    {"name": "box", "description": "box loss gain", "reference": "box"},
    {"name": "scale", "description": "augmentation scale gain", "reference": "scale"},
]

trial_spec = {
    "apiVersion": "batch/v1",
    "kind": "Job",
    "spec": {
        "template": {
            "metadata": {
                "annotations": {
                    # the sidecar never exits, so an injected trial Job hangs
                    "sidecar.istio.io/inject": "false"
                }
            },
            "spec": {
                "restartPolicy": "Never",
                # carries the S3 pod identity association
                "serviceAccountName": "default-editor",
                # the gpu nodepool is tainted workload-class=gpu:NoSchedule, and
                # the nvidia device plugin adds its own taint on top
                "nodeSelector": {"workload-class": "gpu"},
                "tolerations": [
                    {
                        "key": "workload-class",
                        "operator": "Equal",
                        "value": "gpu",
                        "effect": "NoSchedule",
                    },
                    {"key": "nvidia.com/gpu", "operator": "Exists", "effect": "NoSchedule"},
                ],
                "containers": [
                    {
                        "name": "training-container",
                        "image": IMAGE,
                        "command": [
                            "python", "-m", "src.train",
                            # dataset is read-only on the shared volume; the split,
                            # descriptor and outputs must go somewhere writable
                            "--raw=/data/raw",
                            "--processed=/scratch/processed",
                            "--data-yaml=/scratch/data.yaml",
                            "--artifacts=/scratch/artifacts",
                            "--epochs=20",
                            "--imgsz=640",
                            "--device=0",
                            "--project=/scratch/runs",
                            "--lr0=${trialParameters.lr0}",
                            "--batch=${trialParameters.batch}",
                            "--box=${trialParameters.box}",
                            "--scale=${trialParameters.scale}",
                        ],
                        "env": [
                            # the trial logs its own MLflow run while it trains
                            {"name": "MLFLOW_TRACKING_URI",
                             "value": MLFLOW_TRACKING_URI},
                            {"name": "MLFLOW_EXPERIMENT",
                             "value": MLFLOW_EXPERIMENT},
                            {"name": "KATIB_EXPERIMENT", "value": EXPERIMENT_NAME},
                            # the pod name is the Katib trial name, and becomes
                            # the MLflow run name
                            {
                                "name": "POD_NAME",
                                "valueFrom": {
                                    "fieldRef": {"fieldPath": "metadata.name"}
                                },
                            },
                        ],
                        "volumeMounts": [
                            # every parallel trial mounts the same dataset
                            {"name": "dataset", "mountPath": "/data", "readOnly": True},
                            {"name": "scratch", "mountPath": "/scratch"},
                            # Dataloader workers share memory through /dev/shm.
                            {"name": "dshm", "mountPath": "/dev/shm"},
                        ],
                        # g5.xlarge is 4 vCPU / 16Gi / 1 A10G; leave headroom for
                        # the kubelet or the pod will not fit
                        "resources": {
                            "requests": {"cpu": "3", "memory": "12Gi", "nvidia.com/gpu": "1"},
                            "limits": {"cpu": "3", "memory": "12Gi", "nvidia.com/gpu": "1"},
                        },
                    }
                ],
                "volumes": [
                    {
                        "name": "dataset",
                        "persistentVolumeClaim": {
                            "claimName": "dataset-raw",
                            "readOnly": True,
                        },
                    },
                    {"name": "scratch", "emptyDir": {}},
                    # medium=Memory counts against the container memory limit,
                    # so this is carved out of the 12Gi, not added to it
                    {"name": "dshm", "emptyDir": {"medium": "Memory", "sizeLimit": "8Gi"}},
                ],
            },
        }
    },
}

trial_template = V1beta1TrialTemplate(
    primary_container_name="training-container",
    trial_parameters=trial_parameters,
    trial_spec=trial_spec,
    retain=True,  # keep finished trial pods so their logs stay readable
)

print("image      ", IMAGE)
print("parameters ", [p["name"] for p in trial_parameters])
print("dataset     pvc/dataset-raw -> /data (read-only)")
print("scheduling  workload-class=gpu, 1x nvidia.com/gpu per trial")

---

## Submit

- `maxTrialCount`: the budget.
- `parallelTrialCount`: how many GPU nodes Karpenter runs at once.


In [ ]:
metadata = V1ObjectMeta(
    name=EXPERIMENT_NAME,
    namespace=NAMESPACE,
)

experiment = V1beta1Experiment(
    api_version="kubeflow.org/v1beta1",
    kind="Experiment",
    metadata=metadata,
    spec=V1beta1ExperimentSpec(
        objective=objective,
        algorithm=algorithm,
        parameters=parameters,
        trial_template=trial_template,
        max_trial_count=12,
        parallel_trial_count=2,  # = concurrent g5.xlarge nodes
        max_failed_trial_count=3,
    ),
)

client = KatibClient(namespace=NAMESPACE)

client.create_experiment(experiment, namespace=NAMESPACE)

print("submitted  ", EXPERIMENT_NAME)

### Monitor progress

Log progress in noetbook.


In [ ]:
import time


def experiment_condition(exp):
    """Latest condition type on the experiment, or '' before the controller acts."""
    conditions = (exp.status.conditions if exp.status else None) or []
    return conditions[-1].type if conditions else ""


def count(status, field):
    """status is absent until the controller first reconciles the experiment."""
    return getattr(status, field, None) or 0 if status else 0


while True:
    exp = client.get_experiment(name=EXPERIMENT_NAME, namespace=NAMESPACE)
    status = experiment_condition(exp)
    st = exp.status

    print(
        f"{status or 'Pending':12} "
        f"trials: {count(st, 'trials')} "
        f"succeeded {count(st, 'trials_succeeded')} "
        f"running {count(st, 'trials_running')} "
        f"failed {count(st, 'trials_failed')}"
    )

    if status in ("Succeeded", "Failed"):
        break
    time.sleep(60)

---

## Results

Get the optimal assignment Katib converged on.


In [ ]:
best = client.get_optimal_hyperparameters(
    name=EXPERIMENT_NAME, namespace=NAMESPACE
)

print("best objective")
for m in best.observation.metrics:
    print(f"  {m.name:12} {m.latest}")

print("\nbest parameters")
for a in best.parameter_assignments:
    print(f"  {a.name:8} {a.value}")

In [ ]:
import pandas as pd

from kubeflow.katib.constants import constants

response = client.custom_api.list_namespaced_custom_object(
    constants.KUBEFLOW_GROUP,
    constants.KATIB_VERSION,
    namespace=NAMESPACE,
    plural=constants.TRIAL_PLURAL,
    label_selector=f"{constants.EXPERIMENT_LABEL}={EXPERIMENT_NAME}",
)

rows = []
for t in response["items"]:
    status = t.get("status", {})
    conditions = status.get("conditions") or []

    row = {
        "trial": t["metadata"]["name"],
        "status": conditions[-1]["type"] if conditions else "Pending",
    }
    for a in t["spec"].get("parameterAssignments") or []:
        row[a["name"]] = float(a["value"])
    for m in (status.get("observation") or {}).get("metrics") or []:
        row[m["name"]] = float(m["latest"])
    rows.append(row)

df = pd.DataFrame(rows).sort_values("mAP50-95", ascending=False, na_position="last")
df

---

## Runs in MLflow

The trials log themselves while they run, so this only reads back what they
wrote. One MLflow run per trial, named after the trial pod.

Each run holds the hyperparameters as params, a per-epoch curve for the
ultralytics metrics, the final validation pass as `val_*`, and the system
metrics sampled once a minute for the g5.xlarge the trial ran on.


In [ ]:
mlflow_client = MlflowClient()
experiment_id = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT).experiment_id

runs = mlflow_client.search_runs(
    [experiment_id],
    filter_string=f"tags.katib_experiment = '{EXPERIMENT_NAME}'",
    order_by=[f"metrics.`val_{objective.objective_metric_name}` DESC"],
)
print(f"{len(runs)} runs logged by the trials\n")

for r in runs[:5]:
    system = [k for k in r.data.metrics if k.startswith("system/")]
    epochs = len(mlflow_client.get_metric_history(
        r.info.run_id, objective.objective_metric_name))
    print(f"{r.data.tags.get('trial', r.info.run_name):42} "
          f"val={r.data.metrics.get(f'val_{objective.objective_metric_name}', float('nan')):.4f} "
          f"epochs={epochs:3} system_series={len(system)}")

# tag the winner so it is findable without re-reading Katib
if runs:
    mlflow_client.set_tag(runs[0].info.run_id, "best", "true")
    print("\ntagged best  ", runs[0].data.tags.get("trial"))
print("view at      ", f"{MLFLOW_TRACKING_URI}/#/experiments/{experiment_id}")

### Which parameters mattered

A flat cloud means the parameter had little effect over this range; a trend
means it is worth searching more finely.


In [ ]:
import matplotlib.pyplot as plt

searched = [p.name for p in parameters]
done = df[df["mAP50-95"].notna()]

fig, axes = plt.subplots(1, len(searched), figsize=(4 * len(searched), 3.5))
for ax, name in zip(axes, searched):
    ax.scatter(done[name], done["mAP50-95"], alpha=0.8)
    ax.set_xlabel(name)
    ax.set_ylabel("mAP50-95")
    ax.grid(alpha=0.3)
fig.tight_layout()

---

## Apply the result

Fold the winning assignment back into `train-job/configs/train.yaml`, then
retrain at full epochs with those values.

The mirrored runs stay in MLflow under the `kubeflow-yolo-plate-mlflow`
experiment after the Katib objects are removed:

```python
client.delete_experiment(name=EXPERIMENT_NAME, namespace=NAMESPACE)
```


In [ ]:
winning = {a.name: float(a.value) for a in best.parameter_assignments}
print("add to train-job/configs/train.yaml:\n")
for key, value in winning.items():
    print(f"{key}: {value:g}")

# mark the winning trial in MLflow so it is findable without re-reading Katib
best_trial = mlflow_client.search_runs(
    [experiment_id],
    filter_string=f"tags.katib_experiment = '{EXPERIMENT_NAME}'",
    order_by=[f"metrics.`{objective.objective_metric_name}` DESC"],
    max_results=1,
)
if best_trial:
    mlflow_client.set_tag(best_trial[0].info.run_id, "best", "true")
    print("\ntagged best  ", best_trial[0].data.tags.get("trial"))